In [11]:
from paper_utils import *
from typing import Tuple, Set

In [5]:
gap_index = 3
time_index = 4
rocks_rows = get_all_abhsvi_rs_lines()

In [14]:
def get_test_gaps(test: str, horizon: int) -> Set[float]:
    result = set()
    min_so_far = None
    name = f"{test}_{horizon}"
    if test[-1] == '_':
        name += "_0"
    name += ".txt"
    f = open(os.path.join("AB-HSVI_NeurIPS_2025", "Results", name), "r")
    lines = f.readlines()

    for (index, line) in enumerate(lines):
        if len(line) == 1:
            break
        tokens = line.split('\t')
        assert(len(tokens) == 5)

        gap = float(tokens[gap_index])
        time = float(tokens[time_index])

        if min_so_far is None:
            min_so_far = gap
            result.add(gap)
        elif gap < min_so_far:
            result.add(gap)
        min_so_far = min(min_so_far, gap)


    f.close()
    return result
def get_all_gaps() -> Set[float]:
    result = set()
    for horizon in range(1, 8):
        for row in rocks_rows:
            gaps = get_test_gaps(row.benchmark, horizon)
            for g in gaps:
                result.add(g)
    return result

In [33]:
def get_gap_finish_time(row: Row, current_gap) -> float:
    test = row.benchmark
    horizon = row.horizon
    name = f"{test}_{horizon}"
    if test[-1] == '_':
        name += "_0"
    name += ".txt"
    f = open(os.path.join("AB-HSVI_NeurIPS_2025", "Results", name), "r")
    lines = f.readlines()
    for (index, line) in enumerate(lines):
        if len(line) == 1:
            break
        tokens = line.split('\t')
        assert(len(tokens) == 5)

        if len(lines[index+1])  < 5:
            return row.time
        gap = float(tokens[gap_index])
        time = float(tokens[time_index])
        if math.isclose(current_gap, gap, abs_tol=1e-6, rel_tol=1e-6) or gap < current_gap:
            return time
    return row.time



In [34]:
def dump_gap_times():
    df_rows = []
    gaps = get_all_gaps()
    for row in rocks_rows:
        df_row = {
            'benchmark': row.benchmark,
            'horizon': row.horizon,
            'total_time': row.time
        }
        for gap in gaps:
            df_row[str(gap)] = get_gap_finish_time(row, gap)
        df_rows.append(df_row)

    df = pd.DataFrame(df_rows)
    df.to_csv(os.path.join("results", "rocks_gaps.csv"))



In [35]:
dump_gap_times()

In [38]:
# check results
our_rows = get_all_our_rs_lines()

for (row_a, row_o) in zip(rocks_rows, our_rows):
    benchmark_name = row_a.benchmark.split(".")[0]
    assert(row_a.benchmark.split(".")[0] == row_o.benchmark.split(".")[0])

    if row_a.time != "timeout" and row_o.time != "timeout":
        assert( math.isclose(row_a.value, row_o.value, rel_tol=1e-6, abs_tol=1e-6))

